# 05 · Interpretabilidad

**Qué hace este notebook:** explica el modelo que eligió `04`. Responde a *por
qué* predice lo que predice, no a *cuánto* acierta -- eso ya está medido.

**Por qué está separado de `04`:** son preguntas distintas sobre objetos
distintos. `04` compara ocho modelos con una métrica; esto examina uno solo, con
métodos que cuestan minutos u horas. Juntos, añadir un noveno modelo a la
comparación obligaría a recalcular SHAP para nada.

**El orden de las secciones no es negociable.** La importancia va antes que los
efectos parciales, porque un PDP de una variable irrelevante es una curva
perfectamente dibujada que no significa nada, y es muy fácil contar una historia
sobre ella.

| | |
|---|---|
| Lee | `models/best_model.txt` y el `.joblib` que nombra, `data/gold/model_matrix.parquet` |
| Escribe | `reports/figures/interpretability/`, `reports/interpretability/importance.csv` |

**Advertencia que atraviesa todo el notebook:** nada de esto es causalidad. Que
el modelo se apoye en la presión no dice que la presión cause el oleaje, dice que
en estos datos la presión es informativa. Con predictores tan correlacionados como
los meteorológicos, el reparto del crédito entre variables de un mismo grupo es en
buena medida arbitrario -- razón por la que la sección 5 mira grupos y no
variables sueltas.

In [ ]:
DATASET = "model_matrix.parquet"

# Partición sobre la que se explica. `test` es la respuesta correcta: interesa
# cómo se comporta el modelo ante datos que no ha visto. Sobre train se explicaría
# también lo que memorizó.
EXPLAIN_ON = "test"

# SHAP escala con filas x variables x árboles. Se submuestrea para que el
# notebook se ejecute en minutos; subirlo para las figuras definitivas.
SAMPLE = 2000

# Cuántas variables llevar a las figuras de detalle.
TOP_N = 12

# Repeticiones de la permutación. Más alto que en 03 porque aquí el resultado se
# publica y hace falta su error estándar.
N_REPEATS = 20

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

from packagename import get_settings, set_seed, setup_logging
from packagename.etl import read_table, write_table
from packagename.viz import COLOR_NAMES, NEUTRALS, apply_style, savefig

setup_logging(level="INFO")
apply_style("paper")

settings = get_settings()
seed = set_seed(settings.random_seed)
settings.paths.ensure()

FIG = "interpretability"
TABLES = settings.paths.reports / FIG
TABLES.mkdir(parents=True, exist_ok=True)

In [ ]:
name = (settings.paths.models / "best_model.txt").read_text().strip()
bundle = joblib.load(settings.paths.models / name)

model = bundle["model"]
predictors = bundle["predictors"]
TARGET = bundle["target"]

matrix = read_table(settings.paths.gold / DATASET).set_index("time").sort_index()
block = matrix[matrix["split"] == EXPLAIN_ON]

features = block[predictors]
observed = block[TARGET]
sample = features.sample(min(SAMPLE, len(features)), random_state=seed).sort_index()

print(f"Modelo: {bundle['name']} ({name})")
print(f"Explicando sobre {EXPLAIN_ON}: {len(features):,} filas, {len(predictors)} predictores")
print(f"Muestra para SHAP: {len(sample):,} filas")

## 1. Importancia por permutación

La medida menos comprometida de todas: se destruye la relación entre una variable
y el objetivo revolviéndola, y se mide cuánto empeora el modelo. No supone nada
sobre su forma funcional y se interpreta en las unidades de la métrica.

Tiene un punto ciego que conviene conocer antes de leer la figura: con dos
variables muy correlacionadas, permutar una deja la otra intacta y el modelo
apenas se degrada, así que **las dos** salen poco importantes aunque el par sea
esencial. Es exactamente el caso de las variables meteorológicas, y es la razón
de la sección 5.

In [ ]:
permutation = permutation_importance(
    model,
    features,
    observed,
    n_repeats=N_REPEATS,
    random_state=seed,
    n_jobs=-1,
    scoring="neg_root_mean_squared_error",
)

importance = pd.DataFrame(
    {"media": permutation.importances_mean, "std": permutation.importances_std},
    index=predictors,
).sort_values("media", ascending=False)

fig, ax = plt.subplots(figsize=(9, 0.3 * min(len(importance), 30) + 1.5))
top = importance.head(30).iloc[::-1]
ax.barh(top.index, top["media"], xerr=top["std"], color=COLOR_NAMES["mint-green"])
ax.axvline(0, color=NEUTRALS["dark_slate"], lw=0.8)
ax.set_xlabel("aumento del RMSE al permutar")
ax.set_title(f"Importancia por permutación ({EXPLAIN_ON})")
savefig(fig, f"{FIG}/permutation_importance.png")

importance.head(TOP_N).round(4)

## 2. SHAP

SHAP reparte la predicción de **cada fila** entre las variables, y esa es la
diferencia con la permutación: no da un número por variable sino una distribución.
De ahí sale lo que ninguna medida global puede dar -- que una variable importe
mucho en unas pocas situaciones y nada en el resto, que es el patrón típico de las
variables que gobiernan los temporales.

`TreeExplainer` calcula los valores exactos para modelos de árboles sin
aproximación por muestreo. Si el modelo elegido en `04` fuese lineal, el
explicador correcto es `LinearExplainer`; el `if` de abajo lo resuelve, porque
usar `TreeExplainer` sobre un modelo lineal falla en lugar de dar un número
silenciosamente malo.

In [ ]:
if hasattr(model, "coef_"):
    explainer = shap.LinearExplainer(model, sample)
else:
    explainer = shap.TreeExplainer(model)

explanation = explainer(sample)
values = pd.DataFrame(explanation.values, index=sample.index, columns=predictors)

# El SHAP medio absoluto es la versión global de la explicación local. Comparado
# con la permutación es más estable, porque no depende de un remuestreo.
shap_importance = values.abs().mean().sort_values(ascending=False)
shap_importance.head(TOP_N).round(4).to_frame("shap_medio_abs")

In [ ]:
# El beeswarm es la figura central del notebook: cada punto es una fila, la
# posición horizontal su contribución a la predicción y el color el valor de la
# variable. Una nube que se extiende sólo hacia la derecha en su extremo caliente
# es una variable que sólo actúa cuando es alta.
shap.plots.beeswarm(explanation, max_display=TOP_N, show=False)
figure = plt.gcf()
figure.set_size_inches(9, 0.4 * TOP_N + 1.5)
figure.suptitle(f"SHAP: contribución por observación ({EXPLAIN_ON})")
savefig(figure, f"{FIG}/shap_beeswarm.png")

In [ ]:
# Dependencia SHAP: el efecto de cada variable tal como el modelo lo aprendió,
# sin suponer forma funcional. La dispersión vertical a un mismo valor de x es
# interacción con otras variables, y por eso esta figura dice más que un PDP.
leading = shap_importance.index[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, variable in zip(np.ravel(axes), leading, strict=True):
    scatter = ax.scatter(
        sample[variable],
        values[variable],
        c=observed.loc[sample.index],
        s=6,
        alpha=0.5,
        cmap="viridis",
    )
    ax.axhline(0, color=NEUTRALS["dark_slate"], lw=0.8)
    ax.set_xlabel(variable)
    ax.set_ylabel("valor SHAP")
fig.colorbar(scatter, ax=axes, label=f"{TARGET} observado")
fig.suptitle("Dependencia SHAP de los predictores dominantes")
savefig(fig, f"{FIG}/shap_dependence.png")

### 2.1 Casos concretos

Un `waterfall` sobre las observaciones extremas. Es la comprobación de sensatez
que ninguna figura agregada sustituye: si el modelo acierta un temporal
apoyándose en variables que físicamente no tienen nada que ver, el resultado
global es bueno por la razón equivocada, y esto es lo único que lo revela.

In [ ]:
extremes = observed.loc[sample.index].nlargest(2).index
for position, moment in enumerate(extremes):
    row = sample.index.get_loc(moment)
    shap.plots.waterfall(explanation[row], max_display=TOP_N, show=False)
    figure = plt.gcf()
    figure.set_size_inches(9, 0.35 * TOP_N + 2)
    figure.suptitle(f"{moment}: {TARGET} observado = {observed.loc[moment]:.2f}")
    savefig(figure, f"{FIG}/shap_waterfall_{position}.png")

## 3. Efectos parciales

**PDP e ICE.** El PDP es el efecto medio de una variable manteniendo las demás
fijas; las líneas ICE son ese mismo efecto para observaciones individuales.
Mirarlas juntas es lo que evita el error más frecuente: si las ICE no son
paralelas, el promedio del PDP resume comportamientos que se cancelan y la curva
media describe un régimen que no existe en ningún caso concreto.

**El problema del PDP**, y no es menor aquí: para calcularlo se evalúa el modelo
sobre combinaciones que no ocurren nunca -- presión alta con viento de temporal,
por ejemplo. Con predictores correlacionados eso es la mayor parte de la curva, y
el modelo extrapolando fuera de su dominio no informa de nada. De ahí la sección
3.1.

In [ ]:
chosen = importance.index[:6].tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
PartialDependenceDisplay.from_estimator(
    model,
    sample,
    features=chosen,
    kind="both",  # PDP y las ICE bajo él
    subsample=60,
    ax=np.ravel(axes),
    random_state=seed,
    ice_lines_kw={"alpha": 0.15, "lw": 0.5},
    pd_line_kw={"color": COLOR_NAMES["crimson"], "lw": 2},
)
fig.suptitle("PDP (rojo) sobre curvas ICE: si no son paralelas, hay interacción")
savefig(fig, f"{FIG}/pdp_ice.png")

### 3.1 ALE

El *accumulated local effects* arregla el defecto del PDP: en lugar de evaluar el
modelo en combinaciones inventadas, acumula el efecto *local* dentro de cada
intervalo, usando sólo las observaciones que realmente caen en él. Con variables
correlacionadas es la curva en la que hay que fiarse, y comparar ALE con PDP es
informativo por sí mismo: donde discrepan, el PDP estaba extrapolando.

Se implementa aquí porque `sklearn` no lo incluye. Son quince líneas y evita una
dependencia más.

In [ ]:
# -> src/packagename/models/explain.py si acaba usándose en más de un sitio.
def accumulated_local_effects(
    estimator, frame: pd.DataFrame, variable: str, bins: int = 20
) -> pd.Series:
    edges = np.unique(np.quantile(frame[variable], np.linspace(0, 1, bins + 1)))
    # Cada fila se evalúa dos veces, en los bordes de *su* intervalo: la
    # diferencia es el efecto local, y sólo usa combinaciones observadas.
    position = np.clip(np.searchsorted(edges, frame[variable], side="left") - 1, 0, len(edges) - 2)

    lower, upper = frame.copy(), frame.copy()
    lower[variable] = edges[position]
    upper[variable] = edges[position + 1]
    local = estimator.predict(upper) - estimator.predict(lower)

    per_bin = (
        pd.Series(local).groupby(position).mean().reindex(range(len(edges) - 1), fill_value=0.0)
    )
    effect = per_bin.cumsum()
    centres = (edges[:-1] + edges[1:]) / 2
    # Se centra en cero para que la curva se lea como desviación respecto a la
    # predicción media, igual que un PDP centrado.
    return pd.Series(effect.to_numpy() - effect.mean(), index=centres, name=variable)


fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, variable in zip(np.ravel(axes), chosen, strict=True):
    ale = accumulated_local_effects(model, sample, variable)
    ax.plot(ale.index, ale.to_numpy(), color=COLOR_NAMES["evergreen"], lw=1.6)
    ax.axhline(0, color=NEUTRALS["dark_slate"], lw=0.8)
    # El rug marca dónde hay datos: los tramos vacíos de la curva son extrapolación.
    ax.plot(
        sample[variable],
        np.full(len(sample), ax.get_ylim()[0]),
        "|",
        color=NEUTRALS["light"],
        ms=4,
        alpha=0.3,
    )
    ax.set_xlabel(variable)
    ax.set_ylabel("ALE")
fig.suptitle("Efectos locales acumulados (robustos a la correlación entre predictores)")
savefig(fig, f"{FIG}/ale.png")

## 4. Interacciones

Los valores SHAP permiten medir interacción sin ajustar nada más: si el efecto de
una variable depende del valor de otra, sus contribuciones covarían. Es una
aproximación barata a la matriz de interacción SHAP completa, que para un modelo
de cientos de árboles rara vez cabe en memoria.

In [ ]:
leading_pairs = shap_importance.index[:TOP_N]
interaction = values[leading_pairs].corr().abs()
np.fill_diagonal(interaction.to_numpy(), np.nan)

fig, ax = plt.subplots(figsize=(8, 6.5))
mesh = ax.pcolormesh(interaction.to_numpy(), cmap="magma_r", vmin=0, vmax=1)
ax.set_xticks(np.arange(len(leading_pairs)) + 0.5, leading_pairs, rotation=90)
ax.set_yticks(np.arange(len(leading_pairs)) + 0.5, leading_pairs)
ax.invert_yaxis()
ax.grid(visible=False)
ax.set_title("Covariación entre contribuciones SHAP")
fig.colorbar(mesh, ax=ax)
savefig(fig, f"{FIG}/shap_interaction.png")

pairs = (
    interaction.where(np.triu(np.ones(interaction.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
pairs.head(10).round(3).to_frame("|corr| entre SHAP")

## 5. Vuelta a la selección de variables

La comparación que cierra el círculo. `03` eligió variables con criterios
calculados sobre modelos rápidos y sin ajustar; aquí se tiene la importancia del
modelo definitivo, medida sobre datos que nunca vio. Las discrepancias son el
resultado interesante, y hay dos tipos:

- **Alta en `03`, baja aquí:** la variable era redundante. Los criterios
  univariantes la premiaron por su correlación con el objetivo, pero el modelo
  final obtiene la misma información de otra.
- **Baja en `03`, alta aquí:** la variable sólo actúa en interacción, y ningún
  criterio univariante podía verla. Es el caso que justifica no haber cortado el
  ranking más arriba.

Ojo con la conclusión fácil: esto **no** es motivo para volver a `03`, quitar las
variables que aquí salen bajas y reentrenar. Eso usaría la importancia medida en
test para decidir qué entra en el modelo, que es fuga de información por la puerta
de atrás. Lo que se puede hacer con esta tabla es documentar y, si el cambio
parece merecerlo, rehacer la selección **con validación cruzada** y volver a `04`
desde el principio.

In [ ]:
consensus = read_table(settings.paths.reports / "selection" / "consensus_ranking.csv").set_index(
    "variable"
)

comparison = pd.DataFrame(
    {
        "rango_seleccion_03": consensus["ranking_final"],
        "rango_permutacion_05": importance["media"].rank(ascending=False),
        "rango_shap_05": shap_importance.rank(ascending=False),
    }
).dropna()
comparison["desplazamiento"] = comparison["rango_seleccion_03"] - comparison["rango_shap_05"]

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(
    comparison["rango_seleccion_03"],
    comparison["rango_shap_05"],
    s=25,
    color=COLOR_NAMES["cornflower blue"],
)
limit = comparison[["rango_seleccion_03", "rango_shap_05"]].to_numpy().max()
ax.plot([1, limit], [1, limit], color=NEUTRALS["light"], ls="--")
for variable, row in comparison.iterrows():
    if abs(row["desplazamiento"]) > 5:
        ax.annotate(str(variable), (row["rango_seleccion_03"], row["rango_shap_05"]), fontsize=7)
ax.set_xlabel("rango en la selección (03)")
ax.set_ylabel("rango por SHAP en el modelo final (05)")
ax.set_title("Lo que la selección predijo frente a lo que el modelo usó")
savefig(fig, f"{FIG}/ranking_shift.png")

comparison.reindex(comparison["desplazamiento"].abs().sort_values(ascending=False).index).head(15)

In [ ]:
report = importance.rename(columns={"media": "permutacion", "std": "permutacion_std"}).join(
    [shap_importance.to_frame("shap_medio_abs"), comparison[["rango_seleccion_03"]]]
)
write_table(report.reset_index(names="variable"), TABLES / "importance.csv")
print(TABLES / "importance.csv")

## Conclusiones

A rellenar, y esta vez el destinatario es el capítulo de la tesis:

1. **Qué variables gobiernan el modelo**, con la permutación y el SHAP medio de
   acuerdo. Cuando discrepan, la permutación está diluida por la correlación
   (§1) y el SHAP es la medida a citar.
2. **Qué forma tiene cada efecto** (§3): monótono, con saturación, con umbral. El
   ALE es la curva a publicar si los predictores están correlacionados, que es lo
   normal.
3. **Si los efectos son físicamente plausibles.** Es la comprobación más
   importante de todo el notebook, y la única que no da ninguna figura: un modelo
   con buen RMSE cuyo ALE de presión tiene el signo equivocado está acertando por
   la razón equivocada, y no se puede defender.
4. **Qué interacciones aparecen** (§4) y si tienen sentido físico.
5. **Cómo se comporta en los extremos** (§2.1), que es donde el modelo se juega
   su utilidad.
6. **Qué revela la comparación con `03`** (§5), sin usarla para volver a
   seleccionar sobre test.